In [ ]:
import sys
sys.path.append("../..")

from email_finder import (
    find_emails_batch,
    LeadInput,
    EmailFinderResult,
    load_leads_from_csv,
    load_leads_from_google_sheet,
    export_results,
)
from email_finder.config import Config

config = Config()  # Loads from env
print("Config loaded.")

In [ ]:
# Option A: From CSV
leads = load_leads_from_csv("path/to/needs_enrichment.csv")

# Option B: From Google Sheet
# leads = load_leads_from_google_sheet(
#     "https://docs.google.com/spreadsheets/d/YOUR_SHEET_ID",
#     sheet_name="Needs_Enrichment",
# )

print(f"Loaded {len(leads)} leads")
leads[0] if leads else print("(no leads loaded)")

In [ ]:
results = await find_emails_batch(leads, config)

In [ ]:
status_icon = {
    "verified": "✓",
    "catch_all": "~",
    "unverified": "?",
    "not_found": "✗",
    "invalid": "✗",
}

for lead, result in zip(leads, results):
    icon = status_icon.get(result.status, "?")
    email_str = result.email or "N/A"
    conf = f"{result.confidence:.0%}" if result.confidence else ""
    print(f"[{icon}] {lead.full_name:<30}  {email_str:<40}  {result.status}  {conf}")

In [ ]:
idx = 0  # Change to inspect different leads
lead = leads[idx]
result = results[idx]

print(f"Lead: {lead.full_name}")
print(f"Email: {result.email}  |  Status: {result.status}  |  Confidence: {result.confidence:.0%}")
print(f"Source: {result.source}")
print()
print("Discovery log:")
for entry in result.discovery_log:
    node = entry.get('node', '?')
    res = entry.get('result', {})
    found = res.get('found_email') or res.get('found_emails')
    err = res.get('error')
    print(f"  [{node}]  found={found}  error={err}")

## Cell 7: Quick Scenarios Checklist

Use the table below to verify each scenario when running real leads.

| Scenario | How to set up | Expected outcome |
|---|---|---|
| Guest, no email, no domain | name + company only | Perplexity finds domain → pattern gen → Mailin verifies |
| Host, valid email | name + company + email | Flow A confirms; Mailin verifies → "verified" |
| Host, invalid email | name + invalid email | Flow A → Mailin rejects → Flow B finds alternative |
| Guest, LinkedIn known | name + company + linkedin_url | Perplexity still finds domain → patterns |
| Obscure lead | uncommon name + small company | All nodes fail → "not_found" in still_needs CSV |
| Catch-all domain | domain that accepts all | Detected as catch-all; best pattern picked |
